# GOAL
- Matching Stepstone to Pitchbook

# CONFIG

In [15]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import rcParams
from tabulate import tabulate
import itertools

warnings.filterwarnings('ignore')

sys.modules.pop('a20_config.a10_config', None)  # force reload
from a20_config.a10_config import dyr, fi, c, v

print('CWD:', Path.cwd())
assert Path.cwd() == dyr.project_root, "Please set the working directory to the project root"
print("CWD is the project root, GOOD!")

CWD: /sharedata/camm/c_projects/i100_g7__VC_arms
CWD is the project root, GOOD!


Find already matched ones from the pb_ss_investor_match.xlsx

In [16]:
df_aff = pd.read_excel(fi.sp100_20)
df_aff

,pair_id,EntityID,AffiliateID,AffiliateName,Industry,Location,YearFounded,AffiliateType,HQCity,HQState_Province,...,Location_par,YearFounded_par,AffiliateType_par,HQCity_par,HQState_Province_par,HQCountry_par,LastUpdated_par,pair_parent_count,pair_uniq_parent_count,both_sisters_have_same_parent
0,0,166514-86,434783-53,Ballad Ventures,Corporate Venture Capital,"Johnson City, TN",2020.0,Subsidiary,Johnson City,Tennessee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,434783-53,166514-86,Ballad Health,Hospitals/Inpatient Services,"Johnson City, TN",NaN,Parent,Johnson City,Tennessee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,62802-10,57221-29,Bank Austria TFV,Venture Capital,"Vienna, Austria",1996.0,Sister,Vienna,NaN,...,"London, United Kingdom",1945.0,Parent,London,England,United Kingdom,01/30/2023,2.0,1.0,1.0
3,1,57221-29,62802-10,Clinique Saint,Hospitals/Inpatient Services,"Poissy, France",1959.0,Sister,Poissy,NaN,...,"London, United Kingdom",1945.0,Parent,London,England,United Kingdom,01/30/2023,2.0,1.0,1.0
4,2,433943-74,55165-33,University Hospitals,Acquirer,NaN,1866.0,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,247,435249-19,435253-51,Novamind,Venture Capital,"Toronto, Canada",2019.0,Parent,Toronto,Ontario,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
440,249,164588-68,507452-14,Wanyuandia Investment,Corporate Venture Capital,"Hangzhou, China",2021.0,Subsidiary,Hangzhou,Zhejiang,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
441,249,507452-14,164588-68,NaN,NaN,NaN,NaN,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
442,253,166341-79,157276-27,Ora Clinical,Other Healthcare Services,"Andover, MA",1985.0,Parent,Andover,Massachusetts,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# keep only VCs in Enitity IDs
msk = df_aff[v.entity_type].eq(c.vc)
df_vc = df_aff[msk]
df_vc

,pair_id,EntityID,AffiliateID,AffiliateName,Industry,Location,YearFounded,AffiliateType,HQCity,HQState_Province,...,Location_par,YearFounded_par,AffiliateType_par,HQCity_par,HQState_Province_par,HQCountry_par,LastUpdated_par,pair_parent_count,pair_uniq_parent_count,both_sisters_have_same_parent
1,0,434783-53,166514-86,Ballad Health,Hospitals/Inpatient Services,"Johnson City, TN",NaN,Parent,Johnson City,Tennessee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,57221-29,62802-10,Clinique Saint,Hospitals/Inpatient Services,"Poissy, France",1959.0,Sister,Poissy,NaN,...,"London, United Kingdom",1945.0,Parent,London,England,United Kingdom,01/30/2023,2.0,1.0,1.0
4,2,433943-74,55165-33,University Hospitals,Acquirer,NaN,1866.0,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3,56040-94,56084-95,MemorialCare Health System,Clinics/Outpatient Services,"Long Beach, CA",1907.0,Parent,Long Beach,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5,433710-46,50987-17,Mass General Brigham,Hospitals/Inpatient Services,"Boston, MA",1994.0,Parent,Boston,Massachusetts,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434,245,482380-93,159211-27,Edward-Elmhurst Health,Hospitals/Inpatient Services,"Naperville, IL",2013.0,Parent,Naperville,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437,246,435253-51,484448-50,Foundations for Change,Clinics/Outpatient Services,"Peoria, AZ",2017.0,Subsidiary,Peoria,Arizona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
438,247,435253-51,435249-19,Cedar Psychiatry,Clinics/Outpatient Services,"Springville, UT",NaN,Subsidiary,Springville,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
441,249,507452-14,164588-68,NaN,NaN,NaN,NaN,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# keep only unique VC Ids
df_vc = df_vc.drop_duplicates(subset = v.entity_id)
df_vc

,pair_id,EntityID,AffiliateID,AffiliateName,Industry,Location,YearFounded,AffiliateType,HQCity,HQState_Province,...,Location_par,YearFounded_par,AffiliateType_par,HQCity_par,HQState_Province_par,HQCountry_par,LastUpdated_par,pair_parent_count,pair_uniq_parent_count,both_sisters_have_same_parent
1,0,434783-53,166514-86,Ballad Health,Hospitals/Inpatient Services,"Johnson City, TN",NaN,Parent,Johnson City,Tennessee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,57221-29,62802-10,Clinique Saint,Hospitals/Inpatient Services,"Poissy, France",1959.0,Sister,Poissy,NaN,...,"London, United Kingdom",1945.0,Parent,London,England,United Kingdom,01/30/2023,2.0,1.0,1.0
4,2,433943-74,55165-33,University Hospitals,Acquirer,NaN,1866.0,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3,56040-94,56084-95,MemorialCare Health System,Clinics/Outpatient Services,"Long Beach, CA",1907.0,Parent,Long Beach,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5,433710-46,50987-17,Mass General Brigham,Hospitals/Inpatient Services,"Boston, MA",1994.0,Parent,Boston,Massachusetts,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,244,340845-76,127260-28,Cleveland Clinic,Acquirer,"Cleveland, OH",1921.0,Parent,Cleveland,Ohio,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
434,245,482380-93,159211-27,Edward-Elmhurst Health,Hospitals/Inpatient Services,"Naperville, IL",2013.0,Parent,Naperville,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437,246,435253-51,484448-50,Foundations for Change,Clinics/Outpatient Services,"Peoria, AZ",2017.0,Subsidiary,Peoria,Arizona,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
441,249,507452-14,164588-68,NaN,NaN,NaN,NaN,Parent,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
print('# unique VCs:', len(df_vc))

# unique VCs: 106


In [17]:
df_pb_ss = pd.read_excel(fi.sp110_pb_ss)
df_pb_ss

,investorid,simil_investors,gp_id
0,100044-19,0.911261,15692
1,100063-18,0.911261,1873
2,100063-18,0.911261,3650
3,100066-87,0.961464,1363
4,100066-87,1.000000,7570
...,...,...,...
14232,99963-73,0.903224,4159
14233,99963-73,0.936006,10380
14234,99963-73,0.936006,13430
14235,99972-73,0.926492,2621


In [19]:
inv_id = 'investorid'
msk = df_vc[v.entity_id].isin(df_pb_ss[inv_id])
df_vc_1 = df_vc[msk]
df_vc_1

,pair_id,EntityID,AffiliateID,AffiliateName,Industry,Location,YearFounded,AffiliateType,HQCity,HQState_Province,...,Location_par,YearFounded_par,AffiliateType_par,HQCity_par,HQState_Province_par,HQCountry_par,LastUpdated_par,pair_parent_count,pair_uniq_parent_count,both_sisters_have_same_parent
53,30,54810-10,54249-85,UNC REX Healthcare,Laboratory Services (Healthcare),"Raleigh, NC",1894.0,Parent,Raleigh,North Carolina,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,32,52771-15,123845-59,3M (Health Information Systems business),Other Healthcare Services,"Saint Paul, MN",NaN,Sister,Saint Paul,Minnesota,...,"Saint Paul, MN",1902.0,Parent,Saint Paul,Minnesota,United States,01/30/2023,2.0,1.0,1.0
61,35,51450-31,150350-41,IU Health La Porte Hospital,Hospitals/Inpatient Services,"La Porte, IN",1972.0,Parent,La Porte,Indiana,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,47,233900-47,500764-51,Data Test Labs,Laboratory Services (Healthcare),"Plymouth, MI",2017.0,Sister,Plymouth,Michigan,...,"Northbrook, IL",1894.0,Parent,Northbrook,Illinois,United States,05/06/2023,2.0,1.0,1.0
80,48,233163-19,470772-73,Evernorth Health,Other Healthcare Services,"Saint Louis, MO",2020.0,Sister,Saint Louis,Missouri,...,"Bloomfield, CT",1981.0,Parent,Bloomfield,Connecticut,United States,05/06/2023,2.0,1.0,1.0
144,82,228475-27,57814-03,CureDuchenne,Clinics/Outpatient Services,"Newport Beach, CA",2003.0,Parent,Newport Beach,California,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,91,81906-40,482193-37,BetaMed (Poland),Elder and Disabled Care,"Chorzow, Poland",2001.0,Sister,Chorzow,NaN,...,"Paris, France",1902.0,Parent,Paris,NaN,France,01/30/2023,2.0,1.0,1.0
189,108,266690-26,100288-72,Saltzer Medical Group,Clinics/Outpatient Services,"Nampa, ID",1961.0,Sister,Nampa,Idaho,...,"Salt Lake City, UT",1975.0,Parent,Salt Lake City,Utah,United States,05/06/2023,2.0,1.0,1.0
192,110,124225-66,516146-50,Oncocare Medical,Clinics/Outpatient Services,"Singapore, Singapore",2007.0,Sister,Singapore,NaN,...,"Shanghai, China",1994.0,Parent,Shanghai,NaN,China,05/06/2023,2.0,1.0,1.0


In [20]:
msk = df_pb_ss[inv_id].isin(df_vc_1[v.entity_id])
df_pb_ss_1 = df_pb_ss[msk]
df_pb_ss_1

,investorid,simil_investors,gp_id
3871,124225-66,0.903224,5138
6597,228475-27,0.939759,9622
6882,233163-19,0.903224,16012
6939,233900-47,0.920913,1622
6940,233900-47,0.920913,9640
6941,233900-47,0.920913,15399
6942,233900-47,0.920913,15763
6943,233900-47,0.920913,16624
7330,266690-26,0.968932,14129
10314,51450-31,0.947149,1424


In [21]:
# So 9 of them can be matched but it is not good match because it is not a 1-1 to match, I rather do all of them manually instead of
# using the map (only 9 more manual matchiing) - the minimum is .9 with no 1 similiary